# Analysis of Replication dataset and generated one

In [120]:
import os
import glob
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ks_2samp, wasserstein_distance, skew, kurtosis


In [121]:
ref_dir = "../data/replication/"
gen_dir = "../data/generated/replication/REPL_UNCO_final_ep-650_step-0_sde-ve_noise-exponential_lr-1e-04_N-2000_notlinear_layers-4_nheads-8_20260324_1624410/"

# Statistical Analysis of Replication Dataset

It loads the `stats_dict` from the pickle file and it loads every processed CSV into a dicionary keyed by ticker.

In [122]:
FEATURE_COLS = ["log_adj_close"]
DATE_COL     = "date"

In [123]:
def load_processed_universe(data_dir):
    """
    Load every CSV in data_dir into a dict keyed by ticker.
    Files are assumed to have columns DATE_COL and FEATURE_COLS.
    Variable-length series are handled naturally — no padding.
    Ticker is extracted as the first underscore-delimited token of the filename stem.
    """
    csv_files = sorted(Path(data_dir).resolve().glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")
    data = {}
    for fp in csv_files:
        stem   = fp.stem                                # e.g. "AAPL_1980-12-12_2026-03-23"
        ticker = stem.split("_")[0]                     # "AAPL"
        df     = pd.read_csv(fp, parse_dates=[DATE_COL])
        df     = df[[DATE_COL] + FEATURE_COLS].copy()
        df.rename(columns={DATE_COL: "Date"}, inplace=True)
        df["Step"] = np.arange(len(df))
        data[ticker] = df
    return data

## Flatten all stock into one table
We generate an aggregate table over all samples as to perform aggregate statistics and time-specific ones.

In [124]:
def make_long_panel(universe):
    rows = []
    for ticker, df in universe.items():
        d = df.copy()
        d["Ticker"] = ticker
        rows.append(d)
    return pd.concat(rows, axis=0, ignore_index=True)

# Aggregate distribution statistics
* mean
* std
* skewness
* excess kurtosis
* selected quantiles

In [125]:
def summarize_global_distribution(panel):
    summary = {}
    for col in FEATURE_COLS:
        x = panel[col].dropna().to_numpy()
        summary[col] = {
            "mean":            np.mean(x),
            "std":             np.std(x, ddof=0),
            "skew":            skew(x, bias=False),
            "kurtosis_excess": kurtosis(x, fisher=True, bias=False),
            "q01":             np.quantile(x, 0.01),
            "q05":             np.quantile(x, 0.05),
            "median":          np.quantile(x, 0.50),
            "q95":             np.quantile(x, 0.95),
            "q99":             np.quantile(x, 0.99),
        }
    return pd.DataFrame(summary).T

## Two-sample comparison metrics
For each channel it compares reference and generated distributions using:
* KS statistics
* KS p-value
* Wasserstein-1 distance
* moment differences

In [126]:
def compare_global_distributions(panel_ref, panel_gen):
    rows = []
    for col in FEATURE_COLS:
        x = panel_ref[col].dropna().to_numpy()
        y = panel_gen[col].dropna().to_numpy()
        ks_stat, ks_p = ks_2samp(x, y)
        rows.append({
            "feature":             col,
            "ref_mean":            np.mean(x),
            "gen_mean":            np.mean(y),
            "ref_std":             np.std(x, ddof=0),
            "gen_std":             np.std(y, ddof=0),
            "ref_skew":            skew(x, bias=False),
            "gen_skew":            skew(y, bias=False),
            "ref_kurtosis_excess": kurtosis(x, fisher=True, bias=False),
            "gen_kurtosis_excess": kurtosis(y, fisher=True, bias=False),
            "KS_stat":             ks_stat,
            "KS_pvalue":           ks_p,
            "Wasserstein_1":       wasserstein_distance(x, y),
        })
    return pd.DataFrame(rows)

## Pathwise stock-level statistics
For each stock/path it computes:
* mean, variance, skewness, kurtosis of `Close` log-returns
* annualized volatility

This is done to check if aggregating hide a possible path-level mismatch.

In [127]:
def compute_pathwise_stats(universe):
    """
    Per-path moments on log_adj_close.
    T is set to the actual path length of each series.
    GBM estimates are valid when log_adj_close contains log-returns.
    """
    rows = []
    for ticker, df in universe.items():
        r  = df["log_adj_close"].dropna().to_numpy(dtype=np.float64)
        T_ = len(r)
        if T_ < 2:
            continue

        nu_hat = np.sqrt(np.sum(r ** 2) / T_)
        mu_hat = np.sum(r) / T_ + 0.5 * nu_hat ** 2

        rows.append({
            "Ticker":              ticker,
            "n_obs":               T_,
            "mean_log_ret":        np.mean(r),
            "var_log_ret":         np.var(r, ddof=0),
            "skew_log_ret":        skew(r, bias=False),
            "kurt_log_ret_excess": kurtosis(r, fisher=True, bias=False),
            "annualized_vol":      nu_hat,
            "annualized_drift":    mu_hat,
        })
    return pd.DataFrame(rows)

In [128]:
def compare_pathwise_stats(stats_ref, stats_gen):
    cols = [c for c in stats_ref.columns if c not in ("Ticker", "n_obs")]
    rows = []
    for col in cols:
        x = stats_ref[col].dropna().to_numpy()
        y = stats_gen[col].dropna().to_numpy()
        ks_stat, ks_p = ks_2samp(x, y)
        rows.append({
            "metric":        col,
            "ref_mean":      np.mean(x),
            "gen_mean":      np.mean(y),
            "ref_std":       np.std(x, ddof=0),
            "gen_std":       np.std(y, ddof=0),
            "KS_stat":       ks_stat,
            "KS_pvalue":     ks_p,
            "Wasserstein_1": wasserstein_distance(x, y),
        })
    return pd.DataFrame(rows)


## Step-specific statistics
We are comparing the relative step, which is usually better that comparing calendar dates. For each step across all stocks we compute:
* mean
* std
* 5th percentile
*95th percentile

In [129]:
def stepwise_summary(panel):
    out = (
        panel.groupby("Step")[FEATURE_COLS]
        .agg(["mean", "std",
              lambda x: np.quantile(x, 0.05),
              lambda x: np.quantile(x, 0.95)])
    )
    out.columns = [
        f"{col}_{s if isinstance(s, str) else ('q05' if i % 4 == 2 else 'q95')}"
        for i, (col, s) in enumerate(out.columns)
    ]
    out["n_series"] = panel.groupby("Step")[FEATURE_COLS[0]].count().values
    return out.reset_index()

# Date-specific statistics
Suppose that both dataset use same/similar statistics

In [130]:
def datewise_summary(panel):
    out = (
        panel.groupby("Date")[FEATURE_COLS]
        .agg(["mean", "std",
              lambda x: np.quantile(x, 0.05),
              lambda x: np.quantile(x, 0.95)])
    )
    out.columns = [
        f"{col}_{s if isinstance(s, str) else ('q05' if i % 4 == 2 else 'q95')}"
        for i, (col, s) in enumerate(out.columns)
    ]
    out["n_series"] = panel.groupby("Date")[FEATURE_COLS[0]].count().values
    return out.reset_index()

# Distribution plots
We are plotting as to see:
* wider tails
* shifted mean
* over-dispersion
* asymmetry

In [131]:
def plot_feature_histograms(panel_ref, panel_gen, bins=100):
    fig, axes = plt.subplots(len(FEATURE_COLS), 1,
                             figsize=(10, 3 * len(FEATURE_COLS)))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, FEATURE_COLS):
        ax.hist(panel_ref[col].dropna(), bins=bins, density=True,
                alpha=0.5, label="Reference")
        ax.hist(panel_gen[col].dropna(), bins=bins, density=True,
                alpha=0.5, label="Generated")
        ax.set_title(f"Distribution comparison: {col}")
        ax.legend()
    plt.tight_layout(); plt.show()

## ECDF plots
It plots empirical CDFs for each feature. This is visually tied to the KS statistic: the KS distance is the maximum gap between these two curves.

In [132]:
def plot_ecdf_comparison(panel_ref, panel_gen):
    fig, axes = plt.subplots(len(FEATURE_COLS), 1,
                             figsize=(10, 3 * len(FEATURE_COLS)))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, FEATURE_COLS):
        x = np.sort(panel_ref[col].dropna().to_numpy())
        y = np.sort(panel_gen[col].dropna().to_numpy())
        ax.plot(x, np.arange(1, len(x) + 1) / len(x), label="Reference")
        ax.plot(y, np.arange(1, len(y) + 1) / len(y), label="Generated")
        ax.set_title(f"ECDF comparison: {col}")
        ax.legend()
    plt.tight_layout(); plt.show()

## Pathwise metric plots
We are plotting a stock-level distribution of:
* Annualized drift
* annualized volatility
* mean/variance/skew/kurtosis of returns

In [133]:
def plot_pathwise_metric_histograms(stats_ref, stats_gen):
    metric_cols = [c for c in stats_ref.columns if c not in ("Ticker", "n_obs")]
    fig, axes = plt.subplots(len(metric_cols), 1,
                             figsize=(10, 3 * len(metric_cols)))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, metric_cols):
        ax.hist(stats_ref[col].dropna(), bins=50, density=True,
                alpha=0.5, label="Reference")
        ax.hist(stats_gen[col].dropna(), bins=50, density=True,
                alpha=0.5, label="Generated")
        ax.set_title(f"Pathwise distribution: {col}")
        ax.legend()
    plt.tight_layout(); plt.show()


## Time-step graph with uncertainty bands

In [134]:
def plot_stepwise_bands(step_ref, step_gen, feature="log_adj_close"):
    col_mean = f"{feature}_mean"
    col_q05  = f"{feature}_q05"
    col_q95  = f"{feature}_q95"

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(step_ref["Step"], step_ref[col_mean], label="Reference mean")
    ax.fill_between(step_ref["Step"],
                    step_ref[col_q05], step_ref[col_q95], alpha=0.2)
    ax.plot(step_gen["Step"], step_gen[col_mean], label="Generated mean")
    ax.fill_between(step_gen["Step"],
                    step_gen[col_q05], step_gen[col_q95], alpha=0.2)
    ax.set_title(f"Stepwise comparison with 5%–95% band: {feature}")
    ax.set_xlabel("Step"); ax.set_ylabel(feature); ax.legend()
    plt.tight_layout(); plt.show()


# Run the comparison

In [135]:
def evaluate_universes(ref_dir, gen_dir):
    ref_universe = load_processed_universe(ref_dir)
    gen_universe = load_processed_universe(gen_dir)

    panel_ref = make_long_panel(ref_universe)
    panel_gen = make_long_panel(gen_universe)

    global_ref = summarize_global_distribution(panel_ref)
    global_gen = summarize_global_distribution(panel_gen)
    global_cmp = compare_global_distributions(panel_ref, panel_gen)

    path_ref = compute_pathwise_stats(ref_universe)
    path_gen = compute_pathwise_stats(gen_universe)
    path_cmp = compare_pathwise_stats(path_ref, path_gen)

    step_ref = stepwise_summary(panel_ref)
    step_gen = stepwise_summary(panel_gen)

    print("GLOBAL REFERENCE SUMMARY");  display(global_ref)
    print("GLOBAL GENERATED SUMMARY");  display(global_gen)
    print("GLOBAL COMPARISON");         display(global_cmp)
    print("PATHWISE COMPARISON");       display(path_cmp)

    plot_feature_histograms(panel_ref, panel_gen)
    plot_ecdf_comparison(panel_ref, panel_gen)
    plot_pathwise_metric_histograms(path_ref, path_gen)
    plot_stepwise_bands(step_ref, step_gen, feature="log_adj_close")

    return dict(
        panel_ref=panel_ref, panel_gen=panel_gen,
        global_ref=global_ref, global_gen=global_gen, global_cmp=global_cmp,
        path_ref=path_ref, path_gen=path_gen, path_cmp=path_cmp,
        step_ref=step_ref, step_gen=step_gen,
    )

In [136]:
def evaluate_ref_only(ref_dir):
    ref_universe = load_processed_universe(ref_dir)
    panel_ref    = make_long_panel(ref_universe)
    global_ref   = summarize_global_distribution(panel_ref)
    path_ref     = compute_pathwise_stats(ref_universe)
    step_ref     = stepwise_summary(panel_ref)

    print("GLOBAL REFERENCE SUMMARY"); display(global_ref)

    return dict(panel_ref=panel_ref, global_ref=global_ref,
                path_ref=path_ref, step_ref=step_ref)


# Evaluation

In [137]:
results = evaluate_ref_only(ref_dir=ref_dir)


GLOBAL REFERENCE SUMMARY


,mean,std,skew,kurtosis_excess,q01,q05,median,q95,q99
log_adj_close,2.105849,1.94431,-0.384518,-0.051917,-2.692474,-1.277139,2.345282,4.976034,5.86134


In [138]:
results = evaluate_universes(ref_dir=ref_dir, gen_dir=gen_dir)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_26400\2287365289.py:15: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df     = pd.read_csv(fp, parse_dates=[DATE_COL])


ValueError: Missing column provided to 'parse_dates': 'date'